# 1 — Carregamento e Rotulação

Carrega instâncias DIMACS, gera grafos sintéticos e calcula χ(G) via backtracking exato.

**Saída:** nenhum artefato — os grafos são passados diretamente ao notebook 2.

In [2]:
import os
import threading
import warnings
from pathlib import Path

import numpy as np
import networkx as nx

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TIMEOUT_SEC  = 60

np.random.seed(RANDOM_STATE)

ROOT_DIR   = Path('.')
DATA_DIR   = ROOT_DIR / 'data'
DIMACS_DIR = DATA_DIR / 'dimacs'
DIMACS_DIR.mkdir(parents=True, exist_ok=True)

print('Setup concluído.')
print(f'  Instâncias DIMACS: {DIMACS_DIR.resolve()}')

Setup concluído.
  Instâncias DIMACS: C:\Users\Pedro\Desktop\Mestrado\RP\PPGCC21_RP\data\dimacs


## Parser DIMACS

In [3]:
def parse_col_file(filepath):
    """Lê um arquivo .col do benchmark DIMACS e retorna um nx.Graph."""
    G = nx.Graph()
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('c'):
                continue
            if line.startswith('p'):
                parts = line.split()
                n = int(parts[2])
                G.add_nodes_from(range(n))
            elif line.startswith('e'):
                parts = line.split()
                u, v = int(parts[1]) - 1, int(parts[2]) - 1
                if u != v:
                    G.add_edge(u, v)
    return G

## Backtracking para χ(G)

In [4]:
def chromatic_number_backtracking(G, timeout=TIMEOUT_SEC):
    """
    Calcula chi(G) por backtracking exato com timeout via thread.
    Retorna None se o timeout for atingido antes da conclusão.
    """
    G_s = nx.Graph(G)
    G_s.remove_edges_from(nx.selfloop_edges(G_s))
    G_s = nx.convert_node_labels_to_integers(G_s)

    n = G_s.number_of_nodes()
    if n == 0:
        return 0
    if G_s.number_of_edges() == 0:
        return 1

    order = sorted(range(n), key=lambda v: -G_s.degree(v))
    adj   = [list(G_s.neighbors(v)) for v in range(n)]

    try:
        lb = max(len(c) for c in nx.find_cliques(G_s))
    except Exception:
        lb = 1

    greedy = nx.greedy_color(G_s, strategy='largest_first')
    ub     = max(greedy.values()) + 1

    if lb == ub:
        return lb

    result    = [None]
    stop_flag = threading.Event()

    def is_colorable(k):
        colors = [-1] * n
        def bt(idx):
            if stop_flag.is_set():
                return None
            if idx == n:
                return True
            v    = order[idx]
            used = {colors[u] for u in adj[v] if colors[u] >= 0}
            for c in range(k):
                if c not in used:
                    colors[v] = c
                    r = bt(idx + 1)
                    if r is None:
                        return None
                    if r:
                        return True
                    colors[v] = -1
            return False
        return bt(0)

    def compute():
        for k in range(lb, ub):
            if stop_flag.is_set():
                return
            r = is_colorable(k)
            if r is None:
                return
            if r:
                result[0] = k
                return
        result[0] = ub

    t = threading.Thread(target=compute, daemon=True)
    t.start()
    t.join(timeout)

    if t.is_alive():
        stop_flag.set()
        return None

    return result[0]

## Geração de Grafos Sintéticos

In [5]:
def generate_synthetic_graphs(random_state=RANDOM_STATE):
    """Gera famílias diversas de grafos sintéticos com NetworkX."""
    rng    = np.random.default_rng(random_state)
    graphs = []

    for n in range(5, 56, 5):
        for p in [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]:
            for _ in range(3):
                seed = int(rng.integers(100_000))
                graphs.append(('random_er', nx.erdos_renyi_graph(n, p, seed=seed)))

    for n in [8, 12, 16, 20, 24, 30, 40]:
        for d in [2, 3, 4, 6]:
            if d < n and (n * d) % 2 == 0:
                try:
                    seed = int(rng.integers(100_000))
                    graphs.append(('regular', nx.random_regular_graph(d, n, seed=seed)))
                except nx.NetworkXError:
                    pass

    for r in range(2, 9):
        for c in range(2, 9):
            G = nx.convert_node_labels_to_integers(nx.grid_2d_graph(r, c))
            graphs.append(('grid', G))

    for k in range(2, 16):
        graphs.append(('clique', nx.complete_graph(k)))

    for m in range(1, 9):
        for n in range(1, 9):
            graphs.append(('bipartite', nx.complete_bipartite_graph(m, n)))

    for n in range(3, 25):
        graphs.append(('cycle', nx.cycle_graph(n)))
    for n in range(2, 25):
        graphs.append(('path', nx.path_graph(n)))

    graphs += [
        ('named', nx.petersen_graph()),
        ('named', nx.dodecahedral_graph()),
        ('named', nx.icosahedral_graph()),
        ('named', nx.octahedral_graph()),
        ('named', nx.tetrahedral_graph()),
    ]

    for n in range(10, 61, 10):
        for m in [1, 2, 3]:
            for _ in range(3):
                seed = int(rng.integers(100_000))
                graphs.append(('barabasi_albert', nx.barabasi_albert_graph(n, m, seed=seed)))

    for n in [10, 20, 30]:
        for k in [2, 4, 6]:
            for p in [0.1, 0.3, 0.5]:
                seed = int(rng.integers(100_000))
                graphs.append(('watts_strogatz', nx.watts_strogatz_graph(n, k, p, seed=seed)))

    print(f'Grafos sintéticos gerados: {len(graphs)}')
    return graphs


synthetic_graphs = generate_synthetic_graphs()

Grafos sintéticos gerados: 484


## Carregamento DIMACS

Coloque arquivos `.col` em `data/dimacs/`.  
Download: https://mat.tepper.cmu.edu/COLOR/instances.html

In [6]:
dimacs_graphs = []
col_files     = sorted(DIMACS_DIR.glob('*.col'))

if not col_files:
    print(f'Nenhum .col encontrado em {DIMACS_DIR}.')
    print('Continuando apenas com grafos sintéticos.')
else:
    for path in col_files:
        G = parse_col_file(path)
        if G.number_of_nodes() > 0:
            dimacs_graphs.append(('dimacs', G))
    print(f'Instâncias DIMACS carregadas: {len(dimacs_graphs)}')

all_graphs = synthetic_graphs + dimacs_graphs
print(f'Total de instâncias: {len(all_graphs)}')

Nenhum .col encontrado em data\dimacs.
Continuando apenas com grafos sintéticos.
Total de instâncias: 484
